# Data Quality and Discovery

*Profile raw records and explain how data quality changes the meaning of a result.*


## Data Wrangling and Reliable Results

Data wrangling prepares records for a stated analysis by making their structure,
values, and relationships explicit. A table can be readable and still contain
unusable values: `"85"` is numeric text, `"eighty"` needs review, and an age of 400
is outside this case's allowed range. A successful conversion alone does not
establish that a value is correct.

Discovery records what was received before changes are made. The source label
describes where a record came from; it does not establish its reliability.
Keep the original values so that later cleaning decisions can be traced.
These concepts reconstruct the explanations in `4_1.pptx`, slides 4–8.


## Inspect Sources, Fields, and Missing Values

The original discovery example combines five records with database, API, CSV,
and web source labels. A DataFrame exposes the same fields in a single table.
`.isna().sum()` counts missing cells. Sorting source labels makes the display
deterministic.


In [1]:
import pandas as pd

raw_records = [
    {"record_id": 1, "age": 25, "score": "85", "source": "db"},
    {"record_id": 2, "age": 30, "score": 90, "source": "api"},
    {"record_id": 3, "age": None, "score": 78, "source": "csv"},
    {"record_id": 4, "age": 22, "score": "eighty", "source": "web"},
    {"record_id": 5, "age": 400, "score": 88, "source": "csv"},
]
raw_df = pd.DataFrame(raw_records)
print("Received records:")
print(raw_df.to_string(index=False))
print("Source labels:", sorted(raw_df["source"].unique()))
print("Missing cells by field:")
print(raw_df.isna().sum())


Received records:
 record_id   age  score source
         1  25.0     85     db
         2  30.0     90    api
         3   NaN     78    csv
         4  22.0 eighty    web
         5 400.0     88    csv
Source labels: ['api', 'csv', 'db', 'web']
Missing cells by field:
record_id    0
age          1
score        0
source       0
dtype: int64


There are five records and four source labels. Record 3 has the only missing
age. The text `eighty` is present, so a missing-value count alone does not detect
its format problem.


## Separate Missing, Invalid, and Unusual Values

`pd.to_numeric(..., errors="coerce")` produces a numeric inspection view and
marks failed conversions as missing. Compare it with the original column to
identify conversion failures. The age rule of 0–120 is a stated rule for this
prepared case, not a statistical outlier threshold.


In [2]:
numeric_scores = pd.to_numeric(raw_df["score"], errors="coerce")
bad_score_format = raw_df["score"].notna() & numeric_scores.isna()
bad_age_range = raw_df["age"].notna() & ~raw_df["age"].between(0, 120)

quality_report = pd.Series(
    {
        "received_rows": len(raw_df),
        "missing_ages": int(raw_df["age"].isna().sum()),
        "score_format_errors": int(bad_score_format.sum()),
        "age_range_errors": int(bad_age_range.sum()),
    }
)
print("Quality report:")
print(quality_report)
print("Score format error IDs:", raw_df.loc[bad_score_format, "record_id"].tolist())
print("Age range error IDs:", raw_df.loc[bad_age_range, "record_id"].tolist())


Quality report:
received_rows          5
missing_ages           1
score_format_errors    1
age_range_errors       1
dtype: int64
Score format error IDs: [4]
Age range error IDs: [5]


The report separates one missing age, one nonnumeric score, and one age range
error. Record 4 and record 5 require different corrections; neither issue can
be resolved by blindly filling every missing cell.


## State Which Observations Enter a Statistic

The raw age mean includes every non-missing number, including 400. An eligible
age mean uses only values that pass the stated range rule. Filtering for this
calculation does not delete the original rows or repair their ages.


In [3]:
eligible_ages = raw_df.loc[raw_df["age"].between(0, 120), "age"]
print(f"Raw non-missing age mean: {raw_df['age'].mean():.2f}")
print("Eligible age count:", len(eligible_ages))
print(f"Eligible age mean: {eligible_ages.mean():.2f}")
print("Original row count:", len(raw_df))


Raw non-missing age mean: 119.25
Eligible age count: 3
Eligible age mean: 25.67
Original row count: 5


The raw mean is 119.25 years. The three eligible ages, 25, 30, and 22,
have mean 25.67 years. This is a summary of the three eligible observations;
it is not an estimate of the missing or invalid ages.


## Tools and Reproducible Decisions

The tool categories in `4_1.pptx`, slides 15–17, support different working styles.

| Tool category | Typical role |
|---|---|
| Python, R, and SQL | Express repeatable transformations in code or queries |
| Spreadsheets | Inspect and edit small tables |
| Visual preparation tools | Configure and record transformations through an interface |
| Distributed platforms | Process data spread across machines |
| AI-assisted tools | Suggest patterns or possible corrections for review |

Every tool still needs an explicit record key, valid-value rules, and a way to
inspect the result. Re-running a transformation should apply the same rules
to the same input. [Conceptual source used by the slides: IBM](https://www.ibm.com/think/topics/data-wrangling).


Profile the input before changing it. Report missing values, conversion failures, and rule violations separately, and state the denominator behind every summary.
